# 02 — Feature Engineering

Build technical features and the forward-volatility targets. Verify there are no NaNs and inspect which features correlate with future volatility.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

from coinpredictor.data.ohlcv import load_ohlcv
from coinpredictor.features import build_features, feature_columns
from coinpredictor.config import MODEL

feats = build_features(load_ohlcv())
cols = feature_columns(feats)
print(len(cols), 'features; targets:', MODEL.target_col, '/', MODEL.regime_col)
feats[cols].tail()

In [ ]:
# Sanity: no NaNs; regime balance; vol target distribution
print('NaNs:', feats.isna().sum().sum())
print(feats[MODEL.regime_col].value_counts(normalize=True))
feats[MODEL.target_col].describe()

In [ ]:
# Correlation of each feature with forward realized volatility
corr = feats[cols + [MODEL.target_col]].corr()[MODEL.target_col].drop(MODEL.target_col).sort_values()
print(corr.tail(10))

# Volatility clusters: trailing vs forward realized volatility
feats[['realized_vol_trailing', MODEL.target_col]].tail(365).plot(figsize=(10, 4), title='Realized volatility (annualized)')

In [ ]:
# Optional: add Phase 2/3 external features
from coinpredictor.features import build_features_full
full = build_features_full(load_ohlcv(), use_macro=True, use_sentiment=True)
print(full.shape)
feature_columns(full)[-10:]